# Fire Detection using Multi-layer Perceptron (MLP)

## Project Overview
This project implements a custom Multi-layer Perceptron (MLP) neural network to detect the presence or absence of fire using sensor data. The notebook covers data loading, preprocessing, feature engineering (IAQ and eCO2 calculation), model definition, training, and evaluation for a binary classification task.

## Dataset
The dataset is loaded from a CSV file located at https://data.mendeley.com/datasets/48j7mm8k56/1. This dataset is kindly provided by M. Anedda. It includes the following features:
- `temperature`
- `pressure`
- `relative_humidity`
- `resistance_gassensor`
- `label_tag` (original labels indicating fire stages)

## Data Preprocessing and Feature Engineering
1.  **Column Selection**: Only relevant sensor data and the `label_tag` are selected.
2.  **Label Mapping**: The `label_tag` values are mapped to a simplified `fire` variable with two classes:
    - `0`: Absence of fire
    - `1`: Presence of fire
    (Note: There are separate label mappings for different 'Nodes' in the notebook, indicating different sensor profiles or experimental setups, all mapping to binary outcomes).
3.  **IAQ and eCO2 Calculation**: Indoor Air Quality (IAQ) and equivalent CO2 (eCO2) in ppm are calculated based on the `resistance_gassensor`, `relative_humidity`, and `temperature` features.
4.  **Feature Standardization**: The input features (`temperature`, `pressure`, `relative_humidity`, `resistance_gassensor`) are standardized using `StandardScaler`.
5.  **Train-Test Split**: The dataset is split into 70% for training and 30% for testing, with stratification to maintain class proportions.

## Model: Custom Multi-layer Perceptron (MLP)
A custom `Neural_Network` class is implemented from scratch, extending `BaseEstimator` and `ClassifierMixin` from scikit-learn. The model architecture is configurable but typically includes:
-   **Input Layer**: 7 neurons (corresponding to `temperature`, `pressure`, `relative_humidity`, `resistance_gassensor`, `heater_profile_step_index`, `IAQ`, `CO2_eq_ppm`)
-   **Hidden Layer**: 12 neurons
-   **Output Layer**: 2 neurons (for the 2 fire classes: 'No Fire' / 'Fire')
-   **Activation Function**: Sigmoid
-   **Training**: Backpropagation algorithm with a configurable learning rate and number of epochs.

## Training and Evaluation
The MLP model is trained using the preprocessed training data. The training process includes:
-   **Epochs**: 1000 epochs (configurable).
-   **Learning Rate**: 0.01 (configurable).
-   **Error Minimization**: The squared error (MSE) is tracked and plotted over epochs.

After training, the model's performance is evaluated on the test set using various binary classification metrics:
-   **Overall Accuracy**
-   **Precision**
-   **Recall**
-   **F1-score**
-   **Balanced Accuracy**
-   **Confusion Matrix**: Visualizes the model's performance for 'No Fire' vs. 'Fire'.
-   **Prediction Latency**: Measures the time taken for predictions.

## Usage
To run this project, you need to:
1.  Download the dataset from the provided link: `https://data.mendeley.com/datasets/48j7mm8k56/1`.
2.  Upload the downloaded dataset to your Google Drive
3.  **Select and run the label mapping cell specific to the node you are using.** Each 'Node' represents a distinct sensor or experimental configuration. Running the correct cell ensures that the `label_tag` column is appropriately translated to the `fire` classes (0 or 1) relevant to that specific node's data distribution.
4.  Run all other cells sequentially.

## Dependencies
-   `pandas`
-   `numpy`
-   `matplotlib`
-   `seaborn`
-   `scikit-learn`
-   `joblib`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Dataset Overview

In [ ]:
# --- Import libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import warnings
warnings.filterwarnings('ignore')

# Load the dataset from CSV file
dataset = pd.read_csv('/content/drive/MyDrive/Rete_neurale_2c/NODO4/Nodo_4_profilo_1.csv')

# Select only necessary columns
selected_columns = ["temperature", "pressure", "relative_humidity", "resistance_gassensor","heater_profile_step_index", "label_tag"]
dataset = dataset[selected_columns]

## Node 1 Configuration

In [ ]:
# Map label_tags to 0 (absence), 1 (fire)

wanted_labels = [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009]
dataset = dataset[dataset["label_tag"].isin(wanted_labels)]
label_mapping = {
    1001: 0,  # absence of fire
    1002: 1,  # fire
    1003: 1,
    1004: 1,
    1005: 1,
    1006: 1,
    1007: 1,
    1008: 0,
    1009: 0,
}

dataset["fire"] = dataset["label_tag"].map(label_mapping)

## Nodes 2-3-4-5 Configuration

In [ ]:
# Map label_tags to 0 (absence), 1 (fire)

wanted_labels = [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008]
dataset = dataset[dataset["label_tag"].isin(wanted_labels)]
label_mapping = {
    1001: 0,  # absence of fire
    1002: 1,  # fire
    1003: 1,
    1004: 1,
    1005: 1,
    1006: 1,
    1007: 0,
    1008: 0,
}

dataset["fire"] = dataset["label_tag"].map(label_mapping)

## Data Visualization

In [ ]:
# --- IAQ and CO2_eq_ppm Calculation (Optimized for BME688) ---

# NOTE: BME688 has resistance values reaching millions of Ohms.
# We increased R_max from 50k to 5M (5,000,000) to avoid clipping clean air data.

R_max_limit = 5_000_000  # 5 MegaOhm (more realistic for BME688)
R0 = 50_000              # Ω clean baseline (conservative estimate)

import numpy as np

# 1. Prepare variables with correct limits
# Upper clip raised drastically to maintain clean air info
Rgas = dataset['resistance_gassensor'].clip(lower=100, upper=R_max_limit)
RH   = dataset['relative_humidity'].clip(lower=0, upper=100)
Temp = dataset['temperature'].clip(lower=-20, upper=80)

# 2. ────────── Custom IAQ (75% Gas + 25% Humidity) ──────────
# Logic: Low resistance = Gas present = High IAQ (Worsening)
# Formula adapted to the new resistance range
gas_score = 75 * (1 - (np.log10(Rgas) / np.log10(R_max_limit))) # Use log to handle huge range
hum_score = 25 * (1 - np.abs(40 - RH) / 40)

# Final clip to stay within 0-500 range
dataset['IAQ'] = (5 * (gas_score + hum_score)).clip(lower=0, upper=500)

# 3. ────────── eCO2 equivalent (ppm) ──────────
# Multivariate regression (Estimation)
dataset['CO2_eq_ppm'] = (
      400 # Standard atmospheric baseline
    + 1000 * np.log10(R0 / Rgas) # Logarithmic sensitivity to resistance drop
    + 2.0 * (RH - 40)
    - 3.0 * (Temp - 25)
).clip(lower=400, upper=5000)   # Limit to realistic physical values

print("Feature Engineering completed.")
print(dataset[['resistance_gassensor', 'IAQ', 'CO2_eq_ppm']].describe())
dataset.head()

In [ ]:
# Select all numerical columns to plot
selected_columns = ["temperature", "pressure", "relative_humidity", "resistance_gassensor", "IAQ", "CO2_eq_ppm","fire"]
selected_data = dataset[selected_columns]

In [ ]:
# Plot the selected data
selected_data.plot(subplots=True, figsize=(30, 15), sharex=False, sharey=False)
plt.legend(loc='upper right')
plt.show()

## Outlier Removal

In [ ]:
# --- Data Cleaning ---

initial_shape = dataset.shape

# 1. Remove impossible values/sensor errors (Sanity Check)
dataset = dataset[
    (dataset['relative_humidity'] >= 0) & (dataset['relative_humidity'] <= 100) &
    (dataset['temperature'] >= -40) & (dataset['temperature'] <= 125) & # BME688 operating range
    (dataset['resistance_gassensor'] > 0) & # Resistance cannot be negative
    (dataset['pressure'] > 800) & (dataset['pressure'] < 1200) # Reasonable atmospheric range
]

# 2. Remove rows with missing values (NaN) generated by previous calculations
dataset = dataset.dropna()

print(f"Rows removed for physical errors: {initial_shape[0] - dataset.shape[0]}")
print(f"Final dataset shape: {dataset.shape}")

# Quick check: ensure the fire class is still present
print("Class distribution after cleaning:")
print(dataset['fire'].value_counts())

dataset.head()

In [ ]:
# --- Train/Test Split and Scaling ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. FEATURE SELECTION
selected_columns = [
    "temperature",
    "pressure",
    "relative_humidity",
    "resistance_gassensor",
    "heater_profile_step_index",
    "IAQ",
    "CO2_eq_ppm"
]

print("Features used for training:", selected_columns)

X = dataset[selected_columns].values # Convert to numpy array
y = dataset["fire"].values           # Target variable

# 2. SPLIT BEFORE SCALING
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
# 'stratify=y' ensures the same fire % in both train and test sets

# 3. SCALING
sc = StandardScaler()

# Calculate mean/variance ONLY on the training set
train_X = sc.fit_transform(train_X)

# Apply the same transformation to the test set
test_X = sc.transform(test_X)

# Print data shapes for verification
print(f"\nShape of training data : {train_X.shape}, Label: {train_y.shape}")
print(f"Shape of testing data  : {test_X.shape}, Label: {test_y.shape}")

# Visual check: see first 3 scaled samples
print("\nExample of first 3 scaled training samples:")
print(train_X[:3])

## Multi-Layer Perceptron (MLP) Model

In [ ]:

from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import joblib

class Neural_Network(BaseEstimator, ClassifierMixin):
    def __init__(self, params=None):
        self.params = params
        if params is None:
            # Default aggiornati
            self.inputLayer = 7
            self.hiddenLayer = 12
            self.outputLayer = 2  # 0 e 1
            self.learningRate = 0.01
            self.max_epochs = 1000
            self.BiasHiddenValue = 1
            self.BiasOutputValue = 1
        else:
            self.inputLayer = params['InputLayer']
            self.hiddenLayer = params['HiddenLayer']
            self.outputLayer = params['OutputLayer']
            self.learningRate = params['LearningRate']
            self.max_epochs = params['Epocas']
            self.BiasHiddenValue = params['BiasHiddenValue']
            self.BiasOutputValue = params['BiasOutputValue']

        # Attivazione
        self.activation = self._sigmoid
        self.deriv = self._sigmoid_deriv

        # Inizializzazione pesi e bias (Usiamo NumPy nativo per velocità)
        np.random.seed(42) # Per riproducibilità
        self.WEIGHT_hidden = 2 * np.random.random((self.inputLayer, self.hiddenLayer)) - 1
        self.WEIGHT_output = 2 * np.random.random((self.hiddenLayer, self.outputLayer)) - 1

        self.BIAS_hidden = np.full(self.hiddenLayer, self.BiasHiddenValue, dtype=float)
        self.BIAS_output = np.full(self.outputLayer, self.BiasOutputValue, dtype=float)
        self.classes_number = 2

    def _sigmoid(self, x):
        # Clip per evitare overflow esponenziale
        x = np.clip(x, -500, 500)
        return 1 / (1 + np.exp(-x))

    def _sigmoid_deriv(self, x):
        return x * (1 - x)

    def Backpropagation_Algorithm(self, x):
        # Calcolo Delta Output
        ERROR_output = self.output - self.OUTPUT_L2
        DELTA_output = (-1) * ERROR_output * self.deriv(self.OUTPUT_L2)

        # Calcolo Delta Hidden
        delta_hidden = np.dot(self.WEIGHT_output, DELTA_output) * self.deriv(self.OUTPUT_L1)

        # Aggiornamento Pesi
        # Reshape necessari per garantire che il prodotto matriciale sia corretto
        x_reshaped = x.reshape(-1, 1) # Vettore colonna
        delta_hidden_reshaped = delta_hidden.reshape(1, -1) # Vettore riga

        # Update Pesi
        self.WEIGHT_output -= self.learningRate * np.outer(self.OUTPUT_L1, DELTA_output)
        self.WEIGHT_hidden -= self.learningRate * np.outer(x, delta_hidden)

        # Update Bias
        self.BIAS_output -= self.learningRate * DELTA_output
        self.BIAS_hidden -= self.learningRate * delta_hidden


    def show_err_graphic(self, v_erro, v_epoca):
        plt.figure(figsize=(9, 4))
        plt.plot(v_epoca, v_erro, "r-", label="Training Error")
        plt.xlabel("Epochs")
        plt.ylabel("MSE Loss")
        plt.title("Training Progress")
        plt.legend()
        plt.grid(True)
        plt.show()

    def predict(self, X):
        # Metodo standard sklearn per predire
        predictions = []
        forward = self.activation(np.dot(X, self.WEIGHT_hidden) + self.BIAS_hidden)
        forward = self.activation(np.dot(forward, self.WEIGHT_output) + self.BIAS_output)

        for i in forward:
            predictions.append(np.argmax(i))
        return np.array(predictions)

    def fit(self, X, y, batch_size=32):
        count_epoch = 1
        n = len(X)
        epoch_array = []
        error_array = []

        # Assicuriamoci che y sia un array numpy
        y_values = y if isinstance(y, np.ndarray) else y.values

        while count_epoch <= self.max_epochs:
            total_error = 0
            indices = np.random.permutation(n)
            X_shuffled = X[indices]
            y_shuffled = y_values[indices]

            for i in range(0, n, batch_size):
                batch_X = X_shuffled[i:i+batch_size]
                batch_y = y_shuffled[i:i+batch_size]

                for idx, inputs in enumerate(batch_X):
                    self.output = np.zeros(self.classes_number)

                    # Forward
                    self.OUTPUT_L1 = self.activation(np.dot(inputs, self.WEIGHT_hidden) + self.BIAS_hidden)
                    self.OUTPUT_L2 = self.activation(np.dot(self.OUTPUT_L1, self.WEIGHT_output) + self.BIAS_output)

                    # One-hot encoding on the fly
                    target = batch_y[idx]
                    if target == 0:
                        self.output = np.array([1, 0]) # No Fire
                    elif target == 1:
                        self.output = np.array([0, 1]) # Fire

                    # Error accumulation
                    square_error = np.sum(0.5 * (self.output - self.OUTPUT_L2) ** 2)
                    total_error += square_error

                    # Backprop
                    self.Backpropagation_Algorithm(inputs)

            total_error /= n

            if count_epoch % 50 == 0 or count_epoch == 1:
                print(f"Epoch {count_epoch}/{self.max_epochs} - Loss: {total_error:.6f}")
                error_array.append(total_error)
                epoch_array.append(count_epoch)

            count_epoch += 1

        self.show_err_graphic(error_array, epoch_array)
        return self

## Model Training

In [ ]:
# --- NETWORK TRAINING ---

# Define Parameters
params = {
    'InputLayer': 7,      # Temp, Press, Hum, Res, Step, IAQ, CO2
    'HiddenLayer': 12,
    'OutputLayer': 2,
    'Epocas': 1000,        # Number of epochs
    'LearningRate': 0.01, # Learning rate
    'BiasHiddenValue': 1,
    'BiasOutputValue': 1
}

# Create Model Instance
model = Neural_Network(params)

print("Starting training...")
# Start training
model.fit(train_X, train_y)
print("Training completed!")

# --- EVALUATION ---
print("\nEvaluation on Test Set:")
predictions = model.predict(test_X)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

acc = accuracy_score(test_y, predictions)
print(f"Final Accuracy: {acc*100:.2f}%")

print("\nClassification Report:")
print(classification_report(test_y, predictions, target_names=['No Fire', 'Fire']))

# Confusion Matrix
cm = confusion_matrix(test_y, predictions)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import joblib

# Save network parameters and the scaler
model_data = {
    'WEIGHT_hidden': model.WEIGHT_hidden,
    'BIAS_hidden': model.BIAS_hidden,
    'WEIGHT_output': model.WEIGHT_output,
    'BIAS_output': model.BIAS_output,
    'scaler': sc
}
joblib.dump(model_data, '/content/drive/MyDrive/Rete_neurale_2c/modello_incendio.pkl')

In [ ]:
import joblib
model_data = joblib.load('/content/drive/MyDrive/Rete_neurale_2c/modello_incendio.pkl')
print("MEANS:", list(model_data['scaler'].mean_))
print("STD_DEV:", list(model_data['scaler'].scale_))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import numpy as np

# 1. GENERATE PREDICTIONS
predictions = model.predict(test_X)
y_true = test_y if isinstance(test_y, np.ndarray) else test_y.values

# 2. CALCULATE METRICS
hits_count = np.sum(predictions == y_true)
total_samples = len(y_true)
hits_perc = (hits_count / total_samples) * 100
faults_perc = 100 - hits_perc

print(f"Percentages: {hits_perc:.2f}% hits and {faults_perc:.2f}% faults")
print(f"Total samples: {total_samples}")

# 3. PIE CHART (HITS vs FAULTS)
labels = 'Hits', 'Faults'
sizes = [hits_perc, faults_perc]
explode = (0, 0.1)

fig1, ax1 = plt.subplots()
ax1.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%',
        shadow=True, startangle=90, colors=['green', 'red'])
ax1.axis('equal')
plt.title("Overall Performance")
plt.show()

# 4. CALCULATE ACCURACY PER CLASS
cm = confusion_matrix(y_true, predictions)

total_no_fire = cm[0, :].sum()
total_fire = cm[1, :].sum()

acc_no = (cm[0, 0] / total_no_fire * 100) if total_no_fire > 0 else 0
acc_fire = (cm[1, 1] / total_fire * 100) if total_fire > 0 else 0

print(f"- Accuracy No fire: {acc_no:.2f}% ({cm[0,0]}/{total_no_fire})")
print(f"- Accuracy Fire:    {acc_fire:.2f}% ({cm[1,1]}/{total_fire})")

# 5. BAR CHART PER CLASS
names = ["No Fire", "Fire"]
values = [acc_no, acc_fire]
colors = ['orange', 'purple']

plt.figure(figsize=(6, 4))
plt.bar(names, values, color=colors)
plt.ylabel('Accuracy %')
plt.title('Accuracy by Class')
plt.ylim(0, 100)
for i, v in enumerate(values):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')
plt.show()

# 6. CONFUSION MATRIX VISUALIZATION
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['NO FIRE', 'FIRE'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score
import time

# Calculate additional metrics
overall_accuracy = accuracy_score(y_true, predictions)
precision = precision_score(y_true, predictions)
recall = recall_score(y_true, predictions)
f1 = f1_score(y_true, predictions)
balanced_accuracy = balanced_accuracy_score(y_true, predictions)

# ROC-AUC requires probability scores, which our current predict method doesn't provide.
# For now, we will skip ROC-AUC or use a dummy value, or modify the NN class to output probabilities.
# Since the `predict` method currently returns class labels, we cannot directly calculate ROC-AUC without probability scores.
# If you need ROC-AUC, the Neural_Network class's predict method would need to be adjusted to return probabilities.
# For demonstration, I'll calculate it using the predicted labels, which is not ideal but will provide a value.
# In a real scenario, you'd want `model.predict_proba(test_X)[:, 1]`
roc_auc = roc_auc_score(y_true, predictions)

# Calculate Latency
start_time = time.time()
_ = model.predict(test_X) # Run prediction again to measure its time
end_time = time.time()
latency = (end_time - start_time) * 1000 # Latency in milliseconds

print(f"Overall Accuracy: {overall_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"ROC-AUC: {roc_auc:.4f} (Note: Calculated using predicted labels, not probabilities)")
print(f"Prediction Latency: {latency:.2f} ms")